In [ ]:
import pandas as pd
import networkx as nx
from pecanpy import pecanpy as pc # Renamed for clarity
from gensim.models import Word2Vec
import os

# --- Load the graph from edgelist ---
print("Loading graph from edgelist...")
graph_path = "../data/graphs/graph.edgelist"
try:
    G = nx.read_edgelist(graph_path)
    print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
except FileNotFoundError:
    print(f"Error: Graph file not found at {graph_path}. Please check the path.")
    exit()

# --- Generate Node2Vec Embeddings ---
# 1. Initialize the SparseOTF model.
print("Initializing PecanPy SparseOTF model...")
n2v = pc.SparseOTF(
    p=1,
    q=1,
    workers=os.cpu_count(),
    verbose=True,
    
)

# 2. Load the graph into the PecanPy model using read_edg.
print("Loading graph data into PecanPy...")
n2v.read_edg(graph_path, weighted=False, directed=False, delimiter=' ')

# 3. Simulate random walks
print("Simulating random walks...")
walks = n2v.simulate_walks(
    num_walks=10,
    walk_length=30
)

# 4. Generate the embeddings from the walks using gensim.
print("Training Word2Vec model...")
word2vec_model = Word2Vec(
    sentences=walks,
    vector_size=64,
    window=10,
    min_count=1,
    sg=1,
    workers=os.cpu_count(),
    seed=13
)





Loading graph from edgelist...
Loaded graph with 50195 nodes and 136356 edges
Initializing PecanPy SparseOTF model...
Loading graph data into PecanPy...
Simulating random walks...


  0%|          | 0/501950 [00:00<?, ?it/s]

Training Word2Vec model...
Extracting and saving transaction-specific embeddings...


ValueError: invalid literal for int() with base 10: '3457624.0'

In [ ]:
# Retrieve the embeddings from the trained model
embeddings = word2vec_model.wv

# --- Extract and Save Embeddings ---
print("Extracting and saving transaction-specific embeddings...")
txn_embeddings = {node: embeddings[node] for node in G.nodes if node.startswith("txn_") and node in embeddings}
df_embeds = pd.DataFrame.from_dict(txn_embeddings, orient="index")
df_embeds.columns = [f"gnn_embed_{i}" for i in range(df_embeds.shape[1])]


# Assuming 'df_embeds' is the DataFrame from the previous steps
# Clean up and prepare dataframe
df_embeds.index.name = "TransactionID"
df_embeds.reset_index(inplace=True)


df_embeds["TransactionID"] = df_embeds["TransactionID"].str.replace("txn_", "").astype(float).astype(int, errors='ignore')



print(df_embeds.head())

# Save the embeddings
output_path = "../data/processed/gnn_embeddings.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_embeds.to_csv(output_path, index=False)
print(f"Saved GNN embeddings to {output_path}")

Extracting and saving transaction-specific embeddings...
   TransactionID  gnn_embed_0  gnn_embed_1  gnn_embed_2  gnn_embed_3  \
0        3457624    -0.276440    -0.079580    -0.105151     0.192539   
1        3522877    -0.576307     0.325972    -0.420107    -0.536453   
2        3503743    -0.048466    -0.154875    -0.146332     0.369835   
3        3529910    -0.304880     0.094929    -0.333929     0.140805   
4        3332530     0.105582     0.030310     0.213206     0.132421   

   gnn_embed_4  gnn_embed_5  gnn_embed_6  gnn_embed_7  gnn_embed_8  ...  \
0     0.029261     0.727383    -0.166851    -0.042800    -0.307245  ...   
1    -0.473030     0.708215    -0.301411    -0.248844    -0.783823  ...   
2    -0.047022     0.902143    -0.256359     0.043274    -0.241364  ...   
3     0.099985     0.547673    -0.123271    -0.083285    -0.488732  ...   
4    -0.130928     0.577424    -0.094792    -0.031752    -0.671896  ...   

   gnn_embed_54  gnn_embed_55  gnn_embed_56  gnn_embed_57  